In [ ]:
import harbor.analysis.cross_docking as cd
from importlib import reload
reload(cd)

In [ ]:
data = cd.DockingDataModel.deserialize("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/full_cross_dock_v2_combined_results/ALL_1_poses.json")

In [ ]:
ev = cd.Evaluator.from_json_file("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/test_failed_analysis/evaluator_reference_split_comparison_18.json")

In [ ]:
ev.n_bootstraps = 10

In [ ]:
ev.run(data)

In [ ]:
df = data.dataframe

In [ ]:
df.nunique()

In [ ]:
data = cd.DockingDataModel.deserialize("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/full_cross_dock_v2_combined_results/ALL_1_poses.json")
pose_selected = ev.run_pose_selector([data])
pose_selected[0] == data

run_pose_selector is changing the data

In [ ]:
data = cd.DockingDataModel.deserialize("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/full_cross_dock_v2_combined_results/ALL_1_poses.json")

In [ ]:
data.get_groupby_columns()

In [ ]:
# First, let's check if there are any NaN values in these columns
columns_to_check = [
    'RefData_Scaffold_Type', 'TanimotoComboData_Aligned',
    'QueryData_Scaffold_Type', 'Reference_Structure',
    'Reference_Ligand', 'ECFPData_bitsize',
    'Query_Ligand', 'ECFPData_radius'
]

# Check for NaN values
print("NaN counts:")
print(df[columns_to_check].isna().sum())

# Check for exact duplicates in these columns
duplicates = df.duplicated(subset=columns_to_check, keep=False)
print("\nNumber of duplicate rows:", duplicates.sum())

# Show a few examples of duplicated groups
if duplicates.any():
    print("\nExample duplicated groups:")
    print(df[duplicates].groupby(columns_to_check).size().head())

In [ ]:
df = data.dataframe.copy()

In [ ]:
df[df["RefData_Scaffold_Type"].isna()]

In [ ]:
df[df["RefData_Scaffold_Type"].isna()]["Reference_Ligand"].unique()

In [ ]:
df[df["Reference_Ligand"] == "MAT-POS-b3e365b9-4"]

# why isn't chemical similarity data getting added correctly?

In [ ]:
import pandas as pd
import json
scaffold_data = "/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/full_cross_dock_v2_chemical_similarity_data/bemis_murcko_clustering/generic_cluster_labels.csv" 
deduplicate = True
dfms = []

In [ ]:
data = cd.DockingDataModel.deserialize("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/full_cross_dock_v2_combined_results/ALL_1_poses.json")
pose_df = data.dataframe.copy()

In [ ]:
date_dict = "/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/full_cross_dock_v2_cmpd_date_dict/date_dict.json"
structure_cmpd_dict = "/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/full_cross_dock_v2_cmpd_date_dict/structure_to_cmpd_dict.json"

In [ ]:
with open(date_dict, "r") as f:
    date_dict = json.load(f)
missing = [
    ref_structure
    for ref_structure in pose_df.Reference_Structure.unique()
    if ref_structure[:-3] not in date_dict.keys()
]

with open(structure_cmpd_dict, "r") as f:
    structure_cmpd_dict = json.load(f)

In [ ]:
refdf = pd.DataFrame(
        {
            "Reference_Structure": list(pose_df.Reference_Structure.unique()),
            "Reference_Ligand": [
                structure_cmpd_dict.get(x[:-3], None)
                for x in pose_df.Reference_Structure.unique()
            ],
            "Date": [
                date_dict.get(x[:-3], None)
                for x in pose_df.Reference_Structure.unique()
            ],
        }
    )

In [ ]:
pose_df.Reference_Structure.unique()

In [ ]:
refdf.nunique()

## what is in pose_df that is not in refdf?

In [ ]:
pose_df_ref_ligs = pose_df.Reference_Ligand.unique()

In [ ]:
refdf_ref_ligs = refdf.Reference_Ligand.unique()

In [ ]:
pose_df_ref_ligs[~pd.Series(pose_df_ref_ligs).isin(refdf_ref_ligs)]

## were these in the original structure_cmpd_dict?

In [ ]:
structure_cmpd_df = pd.DataFrame(
    {
        "Reference_Structure": list(structure_cmpd_dict.keys()),
        "Reference_Ligand": list(structure_cmpd_dict.values()),
    }
)

In [ ]:
structure_cmpd_df

In [ ]:
missing_ligs = pose_df_ref_ligs[~pd.Series(pose_df_ref_ligs).isin(refdf_ref_ligs)]

In [ ]:
filtered = structure_cmpd_df[structure_cmpd_df.Reference_Ligand.isin(missing_ligs)]

In [ ]:
matching_set = set(filtered.Reference_Ligand.unique())

In [ ]:
matching_set - set(missing_ligs)

In [ ]:
set(missing_ligs) - matching_set

## ok so why aren't those ligands getting added to the right structure?

In [ ]:
refdf = pd.DataFrame(
        {
            "Reference_Structure": list(pose_df.Reference_Structure.unique()),
            "Reference_Ligand": [
                structure_cmpd_dict.get(x[:-3], None)
                for x in pose_df.Reference_Structure.unique()
            ],
            "Date": [
                date_dict.get(x[:-3], None)
                for x in pose_df.Reference_Structure.unique()
            ],
        }
    )

In [ ]:
refdf[refdf.Reference_Ligand.isin(missing_ligs)]

## are any of the structures there?

In [ ]:
all_possible_missing_refs = [ref for ref in filtered.Reference_Structure.unique()]

In [ ]:
all_possible_missing_refs

In [ ]:
refdf['ref'] = refdf.Reference_Structure.apply(lambda x: x[:-3])

In [ ]:
refdf[refdf.ref.isin(all_possible_missing_refs)]

In [ ]:
refdf.Reference_Structure

In [ ]:
refdf

In [ ]:
refdf[refdf.Reference_Structure.isin(filtered.Reference_Structure.unique())]

In [ ]:
pose_df[pose_df.Reference_Ligand.isin(missing_ligs)]

In [ ]:
data_from_pose_df = pose_df[pose_df.Reference_Ligand.isin(missing_ligs)]

In [ ]:
pose_df_missing_refs = pose_df[pose_df.Reference_Ligand.isin(missing_ligs)].Reference_Structure.unique()

In [ ]:
pose_df_missing_refs

In [ ]:
pose_df_missing_refs_df = pose_df[pose_df.Reference_Ligand.isin(missing_ligs)].groupby("Reference_Structure").head(1)

In [ ]:
pose_df_missing_refs_df

In [ ]:
pose_df_missing_refs_df["ref"] = pose_df_missing_refs_df.Reference_Structure.apply(lambda x: x[:-3])

In [ ]:
pose_df_missing_refs_df

In [ ]:
pose_df_missing_refs_short = [p[:-3] for p in pose_df_missing_refs]

### which compounds do these refer to in the structure_cmpd_dict?

In [ ]:
structure_cmpd_df[structure_cmpd_df.Reference_Structure.isin(pose_df_missing_refs_short)]

## combine these dataframes

In [ ]:
structure_cmpd_df_missing = structure_cmpd_df[structure_cmpd_df.Reference_Structure.isin(pose_df_missing_refs_short)]

In [ ]:
pose_df_missing = pose_df[pose_df.Reference_Structure.isin(pose_df_missing_refs)]

In [ ]:
pose_df_missing_refs_df.merge(structure_cmpd_df_missing, left_on="ref", right_on="Reference_Structure", how="inner", suffixes=("_pose_df", "_cmpd_dict"))

In [ ]:
structure_cmpd_df_missing

# Maybe the error is just coming from the original parser?

In [ ]:
from asapdiscovery.data.readers.meta_structure_factory import MetaStructureFactory
from pathlib import Path
fragalysis_dir = Path("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/mpro_fragalysis-04-01-24/")
structure_factory = MetaStructureFactory(structure_dir=None, pdb_file=None,fragalysis_dir=fragalysis_dir,)
complexes = structure_factory.load()

In [ ]:
test_cmpd_name = "ALP-POS-ce760d3f-2"

In [ ]:
print([c for c in complexes if c.ligand.compound_name == test_cmpd_name])

## yes, for some reason these labels are getting pulled in

## check metastructure factory

In [ ]:
metadata_df = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/mpro_fragalysis-04-01-24/metadata.csv")

In [ ]:
metadata_df[metadata_df.alternate_name == test_cmpd_name]

# are there more missing in query ligands?

In [ ]:
additional_query_ligs = set(pose_df.Query_Ligand.unique()) - set(pose_df.Reference_Ligand.unique())
missing_query_ligs = set(pose_df.Reference_Ligand.unique()) - set(pose_df.Query_Ligand.unique())

In [ ]:
additional_query_ligs

In [ ]:
missing_query_ligs